In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
# Thêm thư mục cha (thư mục gốc của project) vào đường dẫn hệ thống
sys.path.append(os.path.abspath('..'))

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Import từ thư mục core
from core.utils import seed_everything, get_device, build_dataloaders, train_one_epoch, evaluate, count_parameters, plot_history
from core.models import BNN

# Configs
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
VAL_RATIO = 0.1
SEED = 42
OUT_DIR = Path("runs_mnist_bnn")
OUT_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device()
print(f"Sử dụng thiết bị: {device}")

Failed to read module file 'C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\re\_casefix.py' for module 're._casefix': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\Repos\BSNN_EdgeAI_Workspace\env_tf\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "d:\Repos\BSNN_EdgeAI_Workspace\env_tf\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1324, in _find_and_load_unl

Failed to read module file 'C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\Repos\BSNN_EdgeAI_Workspace\env_tf\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "d:\Repos\BSNN_EdgeAI_Workspace\env_tf\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1324, in _find_and_load_u

Sử dụng thiết bị: cpu


In [2]:
# Chú ý: Ở đây bạn có thể bật/tắt việc nhị phân hóa ảnh đầu vào rất dễ dàng
train_loader, val_loader, test_loader = build_dataloaders(
    BATCH_SIZE, VAL_RATIO, SEED, binarize_input=False 
)

model = BNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Tổng tham số BNN: {count_parameters(model):,}")

Failed to read module file 'd:\Repos\BSNN_EdgeAI_Workspace\core\utils.py' for module 'core.utils': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\Repos\BSNN_EdgeAI_Workspace\env_tf\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ~~~~~~^^
  File "C:\Users\ADMIN\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 866: character maps to <undefined>
100.0%
100.0%
100.0%
100.0%

Tổng tham số BNN: 54,474


In [3]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train: {train_acc*100:.2f}% | Val: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

if best_state is not None:
    torch.save(best_state, OUT_DIR / "best_model.pt")
    model.load_state_dict(best_state)

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print("-" * 40)
print(f"Test accuracy: {test_acc*100:.2f}%")

Epoch 01/10 | Train: 88.05% | Val: 83.97%
Epoch 02/10 | Train: 90.53% | Val: 91.43%
Epoch 03/10 | Train: 92.77% | Val: 92.12%
Epoch 04/10 | Train: 93.15% | Val: 94.87%
Epoch 05/10 | Train: 93.37% | Val: 91.30%
Epoch 06/10 | Train: 93.83% | Val: 93.85%
Epoch 07/10 | Train: 93.91% | Val: 94.02%
Epoch 08/10 | Train: 94.26% | Val: 93.90%
Epoch 09/10 | Train: 93.97% | Val: 91.82%
Epoch 10/10 | Train: 94.08% | Val: 91.78%
----------------------------------------
Test accuracy: 92.46%


In [4]:
plot_history(history, OUT_DIR)

np.save(OUT_DIR / "train_loss.npy", np.array(history["train_loss"]))
np.save(OUT_DIR / "train_acc.npy", np.array(history["train_acc"]))
np.save(OUT_DIR / "val_loss.npy", np.array(history["val_loss"]))
np.save(OUT_DIR / "val_acc.npy", np.array(history["val_acc"]))